<a href="https://colab.research.google.com/github/Infinity-rudra/DSA-DAA/blob/main/Strassen's_Matrix_Multiplication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Strassen's Matrix Multiplication**

In [ ]:
import numpy as np

def next_power_of_2(n):
    """Find the next power of 2 greater than or equal to n."""
    if n == 0:
        return 1
    if (n & (n - 1)) == 0:
        return n
    return 1 << (n.bit_length())

def pad_matrix(matrix):
    """Pads a matrix with zeros to make it a square of the next power of 2."""
    if isinstance(matrix, list):
        matrix = np.array(matrix)

    n_rows, n_cols = matrix.shape
    n = max(n_rows, n_cols)
    m = next_power_of_2(n)

    # Create a new matrix of zeros with size m x m
    padded_matrix = np.zeros((m, m), dtype=matrix.dtype)

    # Copy the original matrix into the top-left corner
    padded_matrix[:n_rows, :n_cols] = matrix

    return padded_matrix

def strassen_multiply(A, B):
    """
    Performs matrix multiplication using Strassen's algorithm.
    Assumes A and B are square matrices of size 2^n x 2^n.
    """
    n = A.shape[0]

    # Base case: 1x1 matrix
    if n == 1:
        return A * B

    # Split matrices into 4 quadrants
    mid = n // 2
    A11, A12 = A[:mid, :mid], A[:mid, mid:]
    A21, A22 = A[mid:, :mid], A[mid:, mid:]

    B11, B12 = B[:mid, :mid], B[:mid, mid:]
    B21, B22 = B[mid:, :mid], B[mid:, mid:]

    # --- Strassen's 7 recursive calls ---
    M1 = strassen_multiply(A11 + A22, B11 + B22)
    M2 = strassen_multiply(A21 + A22, B11)
    M3 = strassen_multiply(A11, B12 - B22)
    M4 = strassen_multiply(A22, B21 - B11)
    M5 = strassen_multiply(A11 + A12, B22)
    M6 = strassen_multiply(A21 - A11, B11 + B12)
    M7 = strassen_multiply(A12 - A22, B21 + B22)

    # --- Combine results ---
    C11 = M1 + M4 - M5 + M7
    C12 = M3 + M5
    C21 = M2 + M4
    C22 = M1 - M2 + M3 + M6

    # Combine quadrants into a single result matrix
    C = np.zeros((n, n), dtype=A.dtype)
    C[:mid, :mid] = C11
    C[:mid, mid:] = C12
    C[mid:, :mid] = C21
    C[mid:, mid:] = C22

    return C

def strassen(A, B):
    """
    Public-facing function to multiply two matrices (A, B) using Strassen's algorithm,
    handling padding and trimming.
    """
    # Ensure inputs are numpy arrays
    A = np.asarray(A)
    B = np.asarray(B)

    # Check for valid matrix multiplication dimensions
    if A.shape[1] != B.shape[0]:
        raise ValueError(f"Matrix dimension mismatch: A({A.shape}) and B({B.shape})")

    # Store original dimensions for final trimming
    orig_rows_A = A.shape[0]
    orig_cols_B = B.shape[1]

    # Pad matrices to be square and size 2^n x 2^n
    A_padded = pad_matrix(A)
    B_padded = pad_matrix(B)

    # Ensure both are padded to the same size
    n_A = A_padded.shape[0]
    n_B = B_padded.shape[0]

    if n_A > n_B:
        B_padded = pad_matrix(B_padded) # Re-pad B to match A
    elif n_B > n_A:
        A_padded = pad_matrix(A_padded) # Re-pad A to match B

    # Perform the multiplication
    C_padded = strassen_multiply(A_padded, B_padded)

    # Trim the result back to the original expected size (rows_A x cols_B)
    C = C_padded[:orig_rows_A, :orig_cols_B]

    return C

# --- Main execution ---
if __name__ == "__main__":
    # Example 1: 2x2 matrices
    A1 = [[1, 2],
          [3, 4]]

    B1 = [[5, 6],
          [7, 8]]

    print("--- Example 1 (2x2) ---")
    print("Matrix A:\n", np.array(A1))
    print("Matrix B:\n", np.array(B1))

    C1 = strassen(A1, B1)
    print("Result C (Strassen):\n", C1)

    C1_check = np.dot(A1, B1)
    print("Result C (np.dot check):\n", C1_check)
    print("Are results equal?", np.array_equal(C1, C1_check))


    # Example 2: 3x3 matrices (requires padding)
    A2 = [[1, 2, 3],
          [4, 5, 6],
          [7, 8, 9]]

    B2 = [[9, 8, 7],
          [6, 5, 4],
          [3, 2, 1]]

    print("\n--- Example 2 (3x3) ---")
    print("Matrix A:\n", np.array(A2))
    print("Matrix B:\n", np.array(B2))

    C2 = strassen(A2, B2)
    print("Result C (Strassen):\n", C2)

    C2_check = np.dot(A2, B2)
    print("Result C (np.dot check):\n", C2_check)
    print("Are results equal?", np.array_equal(C2, C2_check))

    # Example 3: Non-square matrices (2x3 and 3x2)
    A3 = [[1, 2, 3],
          [4, 5, 6]]

    B3 = [[7, 8],
          [9, 1],
          [2, 3]]

    print("\n--- Example 3 (2x3 * 3x2) ---")
    print("Matrix A:\n", np.array(A3))
    print("Matrix B:\n", np.array(B3))

    C3 = strassen(A3, B3)
    print("Result C (Strassen):\n", C3)

    C3_check = np.dot(A3, B3)
    print("Result C (np.dot check):\n", C3_check)
    print("Are results equal?", np.array_equal(C3, C3_check))

--- Example 1 (2x2) ---
Matrix A:
 [[1 2]
 [3 4]]
Matrix B:
 [[5 6]
 [7 8]]
Result C (Strassen):
 [[19 22]
 [43 50]]
Result C (np.dot check):
 [[19 22]
 [43 50]]
Are results equal? True

--- Example 2 (3x3) ---
Matrix A:
 [[1 2 3]
 [4 5 6]
 [7 8 9]]
Matrix B:
 [[9 8 7]
 [6 5 4]
 [3 2 1]]
Result C (Strassen):
 [[ 30  24  18]
 [ 84  69  54]
 [138 114  90]]
Result C (np.dot check):
 [[ 30  24  18]
 [ 84  69  54]
 [138 114  90]]
Are results equal? True

--- Example 3 (2x3 * 3x2) ---
Matrix A:
 [[1 2 3]
 [4 5 6]]
Matrix B:
 [[7 8]
 [9 1]
 [2 3]]
Result C (Strassen):
 [[31 19]
 [85 55]]
Result C (np.dot check):
 [[31 19]
 [85 55]]
Are results equal? True


In [ ]:
import numpy as np

def next_power_of_2(n):
    """Find the next power of 2 greater than or equal to n."""
    if n == 0:
        return 1
    if (n & (n - 1)) == 0:
        return n
    return 1 << (n.bit_length())

def pad_matrix(matrix):
    """Pads a matrix with zeros to make it a square of the next power of 2."""
    if isinstance(matrix, list):
        matrix = np.array(matrix)

    n_rows, n_cols = matrix.shape
    n = max(n_rows, n_cols)
    m = next_power_of_2(n)

    # Create a new matrix of zeros with size m x m
    padded_matrix = np.zeros((m, m), dtype=matrix.dtype)

    # Copy the original matrix into the top-left corner
    padded_matrix[:n_rows, :n_cols] = matrix

    return padded_matrix

def strassen_multiply(A, B):
    """
    Performs matrix multiplication using Strassen's algorithm.
    Assumes A and B are square matrices of size 2^n x 2^n.
    """
    n = A.shape[0]

    # Base case: 1x1 matrix
    if n == 1:
        return A * B

    # Split matrices into 4 quadrants
    mid = n // 2
    A11, A12 = A[:mid, :mid], A[:mid, mid:]
    A21, A22 = A[mid:, :mid], A[mid:, mid:]

    B11, B12 = B[:mid, :mid], B[:mid, mid:]
    B21, B22 = B[mid:, :mid], B[mid:, mid:]

    # --- Strassen's 7 recursive calls ---
    M1 = strassen_multiply(A11 + A22, B11 + B22)
    M2 = strassen_multiply(A21 + A22, B11)
    M3 = strassen_multiply(A11, B12 - B22)
    M4 = strassen_multiply(A22, B21 - B11)
    M5 = strassen_multiply(A11 + A12, B22)
    M6 = strassen_multiply(A21 - A11, B11 + B12)
    M7 = strassen_multiply(A12 - A22, B21 + B22)

    # --- Combine results ---
    C11 = M1 + M4 - M5 + M7
    C12 = M3 + M5
    C21 = M2 + M4
    C22 = M1 - M2 + M3 + M6

    # Combine quadrants into a single result matrix
    C = np.zeros((n, n), dtype=A.dtype)
    C[:mid, :mid] = C11
    C[:mid, mid:] = C12
    C[mid:, :mid] = C21
    C[mid:, mid:] = C22

    return C

def strassen(A, B):
    """
    Public-facing function to multiply two matrices (A, B) using Strassen's algorithm,
    handling padding and trimming.
    """
    # Ensure inputs are numpy arrays
    A = np.asarray(A)
    B = np.asarray(B)

    # Check for valid matrix multiplication dimensions
    if A.shape[1] != B.shape[0]:
        raise ValueError(f"Matrix dimension mismatch: A({A.shape}) and B({B.shape})")

    # Store original dimensions for final trimming
    orig_rows_A = A.shape[0]
    orig_cols_B = B.shape[1]

    # Pad matrices to be square and size 2^n x 2^n
    A_padded = pad_matrix(A)
    B_padded = pad_matrix(B)

    # Ensure both are padded to the same size
    n_A = A_padded.shape[0]
    n_B = B_padded.shape[0]

    if n_A > n_B:
        B_padded = pad_matrix(B_padded) # Re-pad B to match A
    elif n_B > n_A:
        A_padded = pad_matrix(A_padded) # Re-pad A to match B

    # Perform the multiplication
    C_padded = strassen_multiply(A_padded, B_padded)

    # Trim the result back to the original expected size (rows_A x cols_B)
    C = C_padded[:orig_rows_A, :orig_cols_B]

    return C

# --- Main execution ---
if __name__ == "__main__":
    # Example 1: 2x2 matrices
    A1 = [[1, 2],
          [3, 4]]

    B1 = [[5, 6],
          [7, 8]]

    print("--- Example 1 (2x2) ---")
    print("Matrix A:\n", np.array(A1))
    print("Matrix B:\n", np.array(B1))

    C1 = strassen(A1, B1)
    print("Result C (Strassen):\n", C1)

    C1_check = np.dot(A1, B1)
    print("Result C (np.dot check):\n", C1_check)
    print("Are results equal?", np.array_equal(C1, C1_check))


    # Example 2: 3x3 matrices (requires padding)
    A2 = [[1, 2, 3],
          [4, 5, 6],
          [7, 8, 9]]

    B2 = [[9, 8, 7],
          [6, 5, 4],
          [3, 2, 1]]

    print("\n--- Example 2 (3x3) ---")
    print("Matrix A:\n", np.array(A2))
    print("Matrix B:\n", np.array(B2))

    C2 = strassen(A2, B2)
    print("Result C (Strassen):\n", C2)

    C2_check = np.dot(A2, B2)
    print("Result C (np.dot check):\n", C2_check)
    print("Are results equal?", np.array_equal(C2, C2_check))

    # Example 3: Non-square matrices (2x3 and 3x2)
    A3 = [[1, 2, 3],
          [4, 5, 6]]

    B3 = [[7, 8],
          [9, 1],
          [2, 3]]

    print("\n--- Example 3 (2x3 * 3x2) ---")
    print("Matrix A:\n", np.array(A3))
    print("Matrix B:\n", np.array(B3))

    C3 = strassen(A3, B3)
    print("Result C (Strassen):\n", C3)

    C3_check = np.dot(A3, B3)
    print("Result C (np.dot check):\n", C3_check)
    print("Are results equal?", np.array_equal(C3, C3_check))


--- Example 1 (2x2) ---
Matrix A:
 [[1 2]
 [3 4]]
Matrix B:
 [[5 6]
 [7 8]]
Result C (Strassen):
 [[19 22]
 [43 50]]
Result C (np.dot check):
 [[19 22]
 [43 50]]
Are results equal? True

--- Example 2 (3x3) ---
Matrix A:
 [[1 2 3]
 [4 5 6]
 [7 8 9]]
Matrix B:
 [[9 8 7]
 [6 5 4]
 [3 2 1]]
Result C (Strassen):
 [[ 30  24  18]
 [ 84  69  54]
 [138 114  90]]
Result C (np.dot check):
 [[ 30  24  18]
 [ 84  69  54]
 [138 114  90]]
Are results equal? True

--- Example 3 (2x3 * 3x2) ---
Matrix A:
 [[1 2 3]
 [4 5 6]]
Matrix B:
 [[7 8]
 [9 1]
 [2 3]]
Result C (Strassen):
 [[31 19]
 [85 55]]
Result C (np.dot check):
 [[31 19]
 [85 55]]
Are results equal? True
